# **House Prices — EDA, Data Preparation & Benchmark**

**Dataset:** Ames Housing | Kaggle — Home Data for ML Course  
**Goal:** Predict residential property sale prices using a reproducible 
sklearn Pipeline. Covers EDA, preprocessing and a 3-model benchmark.  
Modelling and hyperparameter optimisation → `02_feature_engineering_and_modelling.ipynb`

---


| Section | Description |
|---|---|
| 1. Setup | Imports, data loading, global constants |
| 2. EDA | Structure, missing values, duplicates, target distribution |
| 3. Data Understanding | Variable-by-variable analysis, transformation registry |
| 4. Data Preparation | Semantic imputation, ColumnTransformer Pipeline |
| 5. Benchmark | Train/test split, 3 models, comparison table, diagnostics |
| 6. Critical Analysis | Overfitting/underfitting, production readiness |
| Kaggle Submission | Pipeline applied to test.csv, submission file |

## **1. Setup & Carga de Datos**

In [1]:
import os
import warnings
import shutil

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from scipy import stats
from scipy.stats import gaussian_kde

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OrdinalEncoder,
    OneHotEncoder,
    RobustScaler,
    StandardScaler,
)
from sklearn import set_config
from xgboost import XGBRegressor
from sklearn.tree import DecisionTreeRegressor

warnings.filterwarnings("ignore")
set_config(display="diagram")
pio.templates.default = "plotly_dark"

pd.options.display.float_format = "{:,.4f}".format
pd.set_option("display.max_columns", 100)

print("Setup OK ✓")

Setup OK ✓


In [2]:
# Load data - download from Kaggle on first run (requires kagglehub)
import kagglehub

from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
DATA_DIR     = PROJECT_ROOT / "data"
REQUIRED  = ["train.csv", "test.csv"]

missing = [f for f in REQUIRED if not os.path.exists(os.path.join(DATA_DIR, f))]
if missing:
    os.makedirs(DATA_DIR, exist_ok=True)
    cache_path = kagglehub.competition_download("home-data-for-ml-course")
    for f in os.listdir(cache_path):
        src, dst = os.path.join(cache_path, f), os.path.join(DATA_DIR, f)
        shutil.copytree(src, dst, dirs_exist_ok=True) if os.path.isdir(src) else shutil.copy2(src, dst)

df_train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
df_test  = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
test_ids = df_test["Id"].copy()

TARGET       = "SalePrice"
TARGET_MODEL = "SalePrice_log"
TARGETS      = [TARGET, TARGET_MODEL]
EXCLUDE      = ["Id"] + TARGETS

print(f"Train : {df_train.shape}")
print(f"Test  : {df_test.shape}")

Train : (1460, 81)
Test  : (1459, 80)


## **2. EDA - Exploratory Data Analysis**

**Goal**: understand the dataset structure *before* making any transformation decisions.  
Covers data types, missing values, duplicates and target distribution.

### **2.1 Estructura general**

In [3]:
print(f"Shape: {df_train.shape[0]} rows x {df_train.shape[1]} columns")
df_train.info()

Shape: 1460 rows x 81 columns
<class 'pandas.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   str    
 3   LotFrontage    1201 non-null   float64
 4   LotArea        1460 non-null   int64  
 5   Street         1460 non-null   str    
 6   Alley          91 non-null     str    
 7   LotShape       1460 non-null   str    
 8   LandContour    1460 non-null   str    
 9   Utilities      1460 non-null   str    
 10  LotConfig      1460 non-null   str    
 11  LandSlope      1460 non-null   str    
 12  Neighborhood   1460 non-null   str    
 13  Condition1     1460 non-null   str    
 14  Condition2     1460 non-null   str    
 15  BldgType       1460 non-null   str    
 16  HouseStyle     1460 non-null   str    
 17  OverallQual    1460 non-null   in

In [4]:
# Descriptive statistics sorted by standard deviation
df_train.describe().T.sort_values("std", ascending=False)

,count,mean,std,min,25%,50%,75%,max
SalePrice,"1,460.0000","180,921.1959","79,442.5029","34,900.0000","129,975.0000","163,000.0000","214,000.0000","755,000.0000"
LotArea,"1,460.0000","10,516.8281","9,981.2649","1,300.0000","7,553.5000","9,478.5000","11,601.5000","215,245.0000"
GrLivArea,"1,460.0000","1,515.4637",525.4804,334.0000,"1,129.5000","1,464.0000","1,776.7500","5,642.0000"
MiscVal,"1,460.0000",43.4890,496.1230,0.0000,0.0000,0.0000,0.0000,"15,500.0000"
BsmtFinSF1,"1,460.0000",443.6397,456.0981,0.0000,0.0000,383.5000,712.2500,"5,644.0000"
BsmtUnfSF,"1,460.0000",567.2404,441.8670,0.0000,223.0000,477.5000,808.0000,"2,336.0000"
TotalBsmtSF,"1,460.0000","1,057.4295",438.7053,0.0000,795.7500,991.5000,"1,298.2500","6,110.0000"
2ndFlrSF,"1,460.0000",346.9925,436.5284,0.0000,0.0000,0.0000,728.0000,"2,065.0000"
Id,"1,460.0000",730.5000,421.6100,1.0000,365.7500,730.5000,"1,095.2500","1,460.0000"
1stFlrSF,"1,460.0000","1,162.6267",386.5877,334.0000,882.0000,"1,087.0000","1,391.2500","4,692.0000"


In [5]:
df_train.describe(include="object").T

,count,unique,top,freq
MSZoning,1460,5,RL,1151
Street,1460,2,Pave,1454
Alley,91,2,Grvl,50
LotShape,1460,4,Reg,925
LandContour,1460,4,Lvl,1311
Utilities,1460,2,AllPub,1459
LotConfig,1460,5,Inside,1052
LandSlope,1460,3,Gtl,1382
Neighborhood,1460,25,NAmes,225
Condition1,1460,9,Norm,1260


### **2.2 Missing Values**

Missing values in this dataset require a fundamental distinction:

- **NaN = truly missing data**: `LotFrontage` (17.7%) — the house has a street frontage 
  but it was not recorded.
- **NaN = feature absent**: `PoolQC` (99.5%), `FireplaceQu` (47.3%), garage/basement 
  columns — NaN means *"this house does not have that feature"*.

Imputing the second group with mean/mode would be incorrect: it would tell the model 
that nearly every house has a high-quality pool when 99.5% have no pool at all.

In [6]:
def missing_values_table(df: pd.DataFrame) -> pd.DataFrame:
    """
    Return a summary DataFrame of columns with missing values,
    sorted by missing percentage descending.
    """
    null_count   = df.isnull().sum()
    null_percent = 100 * null_count / len(df)

    return (
        pd.concat([null_count, null_percent, df.dtypes], axis=1)
        .set_axis(["null_count", "null_%", "dtype"], axis=1)
        .loc[lambda x: x["null_count"] > 0]
        .sort_values("null_%", ascending=False)
    )


df_missing = missing_values_table(df_train)
print(f"Columns with missing values: {len(df_missing)} of {df_train.shape[1]}")
df_missing

Columns with missing values: 19 of 81


,null_count,null_%,dtype
PoolQC,1453,99.5205,str
MiscFeature,1406,96.3014,str
Alley,1369,93.7671,str
Fence,1179,80.7534,str
MasVnrType,872,59.7260,str
FireplaceQu,690,47.2603,str
LotFrontage,259,17.7397,float64
GarageType,81,5.5479,str
GarageYrBlt,81,5.5479,float64
GarageFinish,81,5.5479,str


In [7]:
fig = px.bar(
    df_missing.reset_index(),
    x="null_%",
    y="index",
    orientation="h",
    title="Missing Values by Column",
    labels={"null_%": "% Missing", "index": ""},
    color="null_%",
    color_continuous_scale=["#38bdf8", "#ff5757"],
    text_auto=".1f",
)
fig.update_layout(height=max(500, len(df_missing) * 30), coloraxis_showscale=False)
fig.show()

### **2.3 Duplicates**

In [8]:
n_duplicates = df_train.duplicated().sum()
print(f"Duplicate rows: {n_duplicates}")

Duplicate rows: 0


### **2.4 Target Distribution**

In regression, the target distribution directly affects model performance.

**Why apply `log1p`?**

`SalePrice` has a skewness of ~1.88, a long right tail typical of price data.
This causes two concrete problems:

1. Linear models assume normally distributed residuals. A skewed target produces 
   skewed residuals, making estimates suboptimal.
2. Absolute-scale errors penalise expensive houses disproportionately. A $50k error 
   on a $500k house is proportionally smaller than the same error on a $100k house.

`log1p(x) = log(x + 1)` transforms the target to log scale. The `+1` guards against 
`log(0)`, relevant for area variables that can be exactly zero.

Post-transformation skewness drops to ~0.12. Predictions are reverted with `np.expm1()` 
(exact inverse of `log1p`).

In [9]:
df = df_train.copy()
df[TARGET_MODEL] = np.log1p(df[TARGET])

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        f"SalePrice original (skew={df[TARGET].skew():.2f})",
        f"log1p(SalePrice) (skew={df[TARGET_MODEL].skew():.2f})",
    ],
)

for col_idx, (series, color) in enumerate(
    [(df[TARGET], "#ff5757"), (df[TARGET_MODEL], "#38bdf8")], start=1
):
    kde   = gaussian_kde(series.dropna(), bw_method=0.2)
    x_kde = np.linspace(series.min(), series.max(), 300)

    fig.add_trace(
        go.Histogram(x=series, nbinsx=60, marker_color=color,
                     opacity=0.85, histnorm="probability density", showlegend=False),
        row=1, col=col_idx,
    )
    fig.add_trace(
        go.Scatter(x=x_kde, y=kde(x_kde), mode="lines",
                   line=dict(color="white", width=2), showlegend=False),
        row=1, col=col_idx,
    )

fig.update_layout(title="Target Distribution", height=420, width=900)
fig.show()

In [10]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=["QQ-plot: SalePrice original", "QQ-plot: log1p(SalePrice)"],
)

for col_idx, (series, color) in enumerate(
    [(df[TARGET], "#ff5757"), (df[TARGET_MODEL], "#38bdf8")], start=1
):
    result             = stats.probplot(series)
    theoretical        = result[0][0]
    observed           = result[0][1]
    slope, intercept, _ = result[1]

    fig.add_trace(
        go.Scatter(x=theoretical, y=observed, mode="markers",
                   marker=dict(color=color, size=4, opacity=0.7), showlegend=False),
        row=1, col=col_idx,
    )
    fig.add_trace(
        go.Scatter(
            x=[float(theoretical.min()), float(theoretical.max())],
            y=[slope * theoretical.min() + intercept,
               slope * theoretical.max() + intercept],
            mode="lines",
            line=dict(color="white", width=1.5, dash="dash"),
            showlegend=False,
        ),
        row=1, col=col_idx,
    )

fig.update_layout(title="QQ-plots: Target Normality Check", height=420, width=900)
fig.show()

In [11]:
print("Normality tests — H0: distribution is normal (rejected if p < 0.05)\n")

for series, label in [
    (df[TARGET],       "SalePrice original"),
    (df[TARGET_MODEL], "log1p(SalePrice)  "),
]:
    stat, p = stats.normaltest(series)
    result  = "Normal" if p > 0.05 else "Not normal — H0 rejected"
    print(f"{label}: stat={stat:.4f}  p={p:.6f}  -> {result}")

Normality tests — H0: distribution is normal (rejected if p < 0.05)

SalePrice original: stat=610.8359  p=0.000000  -> Not normal — H0 rejected
log1p(SalePrice)  : stat=25.5072  p=0.000003  -> Not normal — H0 rejected


In [12]:
stat_ks, p_ks = stats.kstest(
    df[TARGET_MODEL], "norm",
    args=(df[TARGET_MODEL].mean(), df[TARGET_MODEL].std()),
)
print(f"Kolmogorov-Smirnov: stat={stat_ks:.4f}  p={p_ks:.6f}")

Kolmogorov-Smirnov: stat=0.0409  p=0.014665


**KS test interpretation:**

KS stat=0.040 means the maximum deviation between the empirical distribution of 
`log1p(SalePrice)` and a perfect normal is only 4%. The transformation is valid 
and statistically justified. Normality assumptions in regression apply to residuals, 
not the target directly.

### **2.5 Classify columns by dtype**

In [13]:
# Classify columns by dtype, excluding targets and ID
numeric_cols, categorical_cols, boolean_cols = [], [], []

for col in df.columns:
    if col in EXCLUDE:
        continue
    if pd.api.types.is_bool_dtype(df[col]):
        boolean_cols.append(col)
    elif pd.api.types.is_numeric_dtype(df[col]):
        numeric_cols.append(col)
    else:
        categorical_cols.append(col)

print(f"Numeric    : {len(numeric_cols)}")
print(f"Categorical: {len(categorical_cols)}")
print(f"Boolean    : {len(boolean_cols)}")

Numeric    : 36
Categorical: 43
Boolean    : 0


### **2.6 Columns (low variance)**

In [14]:
# Constant columns - zero predictive value
constant_cols = [col for col in df.columns if df[col].nunique(dropna=False) == 1]
print(f"Constant columns: {constant_cols or 'None'}")

# Numeric columns with low cardinality - likely ordinal
low_variance_numeric = [col for col in numeric_cols if df[col].nunique() <= 5]
print(f"Low-cardinality numeric (<=5 unique values): {low_variance_numeric}")

Constant columns: None
Low-cardinality numeric (<=5 unique values): ['BsmtFullBath', 'BsmtHalfBath', 'FullBath', 'HalfBath', 'KitchenAbvGr', 'Fireplaces', 'GarageCars', 'YrSold']


## **3. Data Understanding — Transformation Planning**

For each variable: analyse distribution, relationship with target, assign transformation.
Results are stored in a `TransformationRegistry` - the central artefact of this section.

| Transformation | When to use |
|---|---|
| `drop` | ID, target, >80% missing without signal, multicollinearity |
| `passthrough` | Already on correct scale, no transformation needed |
| `standard_scaler` | Continuous numeric, no extreme outliers |
| `robust_scaler` | Numeric with outliers (uses median/IQR instead of mean/std) |
| `ordinal_encoder` | Categorical with semantic order (Ex > Gd > TA > Fa > Po) |
| `one_hot_encoding` | Nominal categorical, no natural order |

In [15]:
class TransformationRegistry:
    """
    Encapsulates the transformation assignment state for all dataset columns.

    Replaces the previous global-state pattern (global transformations +
    set_transformation + sync_transformations) with a single object that
    owns its state and exposes a clean interface.

    The underlying DataFrame schema is:
        column | dtype | is_target | n_unique | pct_null | transformation | note
    """

    def __init__(self, df: pd.DataFrame) -> None:
        self._df = self._build(df)

    def _build(self, df: pd.DataFrame) -> pd.DataFrame:
        return pd.DataFrame({
            "column":         df.columns,
            "dtype":          df.dtypes.values,
            "is_target":      df.columns.isin(TARGETS),
            "n_unique":       [df[c].nunique() for c in df.columns],
            "pct_null":       [round(100 * df[c].isnull().mean(), 2) for c in df.columns],
            "transformation": None,
            "note":           None,
        })

    def set(self, column: str, transformation: str, note: str) -> None:
        """Assign a transformation and note to a single column."""
        mask = self._df["column"] == column
        self._df.loc[mask, "transformation"] = transformation
        self._df.loc[mask, "note"]           = note

    def set_many(self, columns: list[str], transformation: str, note: str) -> None:
        """Assign the same transformation to multiple columns at once."""
        for col in columns:
            self.set(col, transformation, note)

    def sync(self, df: pd.DataFrame) -> None:
        """
        Reconcile registry with the current DataFrame columns.
        Removes dropped columns, adds new ones with no transformation assigned.
        """
        self._df = self._df[self._df["column"].isin(df.columns)].copy()

        new_cols = [c for c in df.columns if c not in self._df["column"].values]
        if new_cols:
            new_rows = pd.DataFrame({
                "column":         new_cols,
                "dtype":          [df[c].dtype for c in new_cols],
                "is_target":      [c in TARGETS for c in new_cols],
                "n_unique":       [df[c].nunique() for c in new_cols],
                "pct_null":       [round(100 * df[c].isnull().mean(), 2) for c in new_cols],
                "transformation": None,
                "note":           None,
            })
            self._df = pd.concat([self._df, new_rows], ignore_index=True)

    def get_columns(self, transformation: str) -> list[str]:
        """Return all columns assigned to a given transformation."""
        return self._df.loc[
            self._df["transformation"] == transformation, "column"
        ].tolist()

    def unassigned(self) -> pd.DataFrame:
        """Return columns with no transformation assigned (excluding targets)."""
        return self._df[
            self._df["transformation"].isna() & ~self._df["is_target"]
        ]

    @property
    def data(self) -> pd.DataFrame:
        """Read-only access to the underlying DataFrame."""
        return self._df.copy()


registry = TransformationRegistry(df)
registry.set("Id",            "drop",   "Unique identifier - no predictive value")
registry.set("SalePrice",     "target", "Original target")
registry.set("SalePrice_log", "target", "Log-transformed target - used for training")

print("TransformationRegistry initialised")
registry.data.head(10)

TransformationRegistry initialised


,column,dtype,is_target,n_unique,pct_null,transformation,note
0,Id,int64,False,1460,0.0000,drop,Unique identifier - no predictive value
1,MSSubClass,int64,False,15,0.0000,None,None
2,MSZoning,str,False,5,0.0000,None,None
3,LotFrontage,float64,False,110,17.7400,None,None
4,LotArea,int64,False,1073,0.0000,None,None
5,Street,str,False,2,0.0000,None,None
6,Alley,str,False,2,93.7700,None,None
7,LotShape,str,False,4,0.0000,None,None
8,LandContour,str,False,4,0.0000,None,None
9,Utilities,str,False,2,0.0000,None,None


### **3.1 Feature Engineering**

Before analysing original variables, we create derived features that capture richer signal:

- `TotalSF`: total living area (basement + 1F + 2F) - more informative than any single floor area
- `HouseAge`: years between build year and sale year - depreciation proxy
- `RemodAge`: years since last remodel - renovation signal
- `WasRemodeled`: binary flag - has the house ever been remodelled?
- `TotalBaths`: weighted bath count (full=1, half=0.5)
- `TotalPorchSF`: combined porch area - replaces 4 low-signal columns

These features reduce dimensionality and increase signal-to-noise ratio.

In [16]:
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Create domain-driven features from raw OHLCV-style housing columns.
    Applied to both train and test to ensure consistency.
    All source columns are filled with 0 before arithmetic to handle
    missing values in test set without leaking train statistics.
    """
    out = df.copy()
    out["TotalSF"]      = out["TotalBsmtSF"].fillna(0) + out["1stFlrSF"].fillna(0) + out["2ndFlrSF"].fillna(0)
    out["HouseAge"]     = out["YrSold"] - out["YearBuilt"]
    out["RemodAge"]     = out["YrSold"] - out["YearRemodAdd"]
    out["WasRemodeled"] = (out["YearRemodAdd"] != out["YearBuilt"]).astype(int)
    out["TotalBaths"]   = (
        out["FullBath"].fillna(0) + out["BsmtFullBath"].fillna(0)
        + 0.5 * (out["HalfBath"].fillna(0) + out["BsmtHalfBath"].fillna(0))
    )
    out["TotalPorchSF"] = (
        out["OpenPorchSF"].fillna(0) + out["EnclosedPorch"].fillna(0)
        + out["3SsnPorch"].fillna(0) + out["ScreenPorch"].fillna(0)
    )
    return out


df      = engineer_features(df)
df_test = engineer_features(df_test)

registry.sync(df)
registry.set_many(
    ["TotalSF", "TotalBaths", "TotalPorchSF"],
    "robust_scaler",
    "Engineered area/count feature — RobustScaler due to potential outliers",
)
registry.set_many(
    ["HouseAge", "RemodAge"],
    "standard_scaler",
    "Engineered age feature — StandardScaler, roughly uniform distribution",
)
registry.set("WasRemodeled", "passthrough", "Binary 0/1 flag — no scaling needed")

print(f"Shape after feature engineering: {df.shape}")

Shape after feature engineering: (1460, 88)


In [17]:
NEW_FEATURES = ["TotalSF", "HouseAge", "RemodAge", "WasRemodeled", "TotalBaths", "TotalPorchSF"]

corr_new = (
    df[NEW_FEATURES + [TARGET]]
    .corr()[TARGET]
    .drop(TARGET)
    .sort_values(ascending=False)
)

fig = px.bar(
    corr_new.reset_index(),
    x=TARGET,
    y="index",
    orientation="h",
    title="Engineered Features — Correlation with SalePrice",
    labels={TARGET: "Pearson r", "index": ""},
    color=TARGET,
    color_continuous_scale=["#ff5757", "#34d399"],
    text_auto=".3f",
)
fig.update_layout(height=350, coloraxis_showscale=False, width=700)
fig.show()

### **3.2 Noise Variables**

Columns with Pearson |r| < 0.05 with SalePrice and extreme distributions
(>95% zeros) are dropped - they add dimensionality without predictive signal.

In [18]:
NOISE_COLS = ["MiscVal", "3SsnPorch", "LowQualFinSF", "PoolArea", "ScreenPorch", "BsmtHalfBath"]

registry.set_many(
    NOISE_COLS,
    "drop",
    "Pearson |r| < 0.05 with SalePrice and extreme zero-inflation — no predictive signal",
)

print(f"Noise columns marked for drop: {NOISE_COLS}")

Noise columns marked for drop: ['MiscVal', '3SsnPorch', 'LowQualFinSF', 'PoolArea', 'ScreenPorch', 'BsmtHalfBath']


### **3.3 Multicollinear Variables**

Columns with high pairwise correlation (r > 0.82) are redundant once
engineered features are in place. Keeping both would introduce multicollinearity
and inflate model variance without adding information.

In [19]:
MULTICOLLINEAR: dict[str, str] = {
    "GarageArea":   "r > 0.88 with GarageCars — redundant",
    "1stFlrSF":     "Captured by TotalSF",
    "TotRmsAbvGrd": "r > 0.82 with GrLivArea — redundant",
    "OpenPorchSF":  "Captured by TotalPorchSF",
    "EnclosedPorch":"Captured by TotalPorchSF",
}

for col, note in MULTICOLLINEAR.items():
    registry.set(col, "drop", note)

print(f"Multicollinear columns dropped: {list(MULTICOLLINEAR.keys())}")

Multicollinear columns dropped: ['GarageArea', '1stFlrSF', 'TotRmsAbvGrd', 'OpenPorchSF', 'EnclosedPorch']


### **3.4 Categorical Variables**

In [20]:
cat_cols_current = [c for c in categorical_cols if c in df.columns]

cat_summary = pd.DataFrame({
    "n_unique":  [df[c].nunique() for c in cat_cols_current],
    "null_%":    [df[c].isnull().mean() * 100 for c in cat_cols_current],
    "top_value": [
        df[c].value_counts().index[0] if df[c].notna().any() else None
        for c in cat_cols_current
    ],
}, index=cat_cols_current).sort_values("null_%", ascending=False)

cat_summary

,n_unique,null_%,top_value
PoolQC,3,99.5205,Gd
MiscFeature,4,96.3014,Shed
Alley,2,93.7671,Grvl
Fence,4,80.7534,MnPrv
MasVnrType,3,59.7260,BrkFace
FireplaceQu,5,47.2603,Gd
GarageFinish,3,5.5479,Unf
GarageQual,5,5.5479,TA
GarageType,6,5.5479,Attchd
GarageCond,5,5.5479,TA


In [21]:
neighborhood_stats = (
    df.groupby("Neighborhood")[TARGET]
    .agg(["median", "count"])
    .sort_values("median", ascending=True)
    .reset_index()
)

fig = px.bar(
    neighborhood_stats,
    x="median",
    y="Neighborhood",
    orientation="h",
    title="Median SalePrice by Neighborhood",
    labels={"median": "Median SalePrice ($)", "Neighborhood": ""},
    color="median",
    color_continuous_scale=["#38bdf8", "#c084fc"],
    text_auto="$,.0f",
)
fig.update_layout(height=500, coloraxis_showscale=False)
fig.show()

In [22]:
# High missingness — NaN means feature absent, not missing data
registry.set_many(
    ["MiscFeature", "Alley", "Fence", "PoolQC"],
    "drop",
    ">80% missing - NaN semantically means absent, low price signal",
)

# Quality ordinals — Po < Fa < TA < Gd < Ex
QUALITY_COLS = [
    "ExterQual", "ExterCond", "HeatingQC", "KitchenQual",
    "FireplaceQu", "GarageQual", "GarageCond", "BsmtQual", "BsmtCond",
]
registry.set_many(QUALITY_COLS, "ordinal_encoder", "Quality ordinal: Po < Fa < TA < Gd < Ex - NaN means absent")

# Other ordinals with domain-specific scales
ORDINAL_ASSIGNMENTS: dict[str, str] = {
    "BsmtExposure": "None < No < Mn < Av < Gd",
    "BsmtFinType1": "None < Unf < LwQ < Rec < BLQ < ALQ < GLQ",
    "BsmtFinType2": "None < Unf < LwQ < Rec < BLQ < ALQ < GLQ",
    "GarageFinish": "None < Unf < RFn < Fin",
    "LotShape":     "IR3 < IR2 < IR1 < Reg",
    "LandSlope":    "Sev < Mod < Gtl",
    "LandContour":  "Low < HLS < Bnk < Lvl",
    "PavedDrive":   "N < P < Y",
    "CentralAir":   "N < Y",
    "Utilities":    "ELO < NoSeWa < NoSewr < AllPub",
}
for col, scale in ORDINAL_ASSIGNMENTS.items():
    registry.set(col, "ordinal_encoder", f"Ordinal: {scale}")

# MSSubClass is int but encodes nominal house types - OHE required
registry.set("MSSubClass", "one_hot_encoding", "Integer but nominal - encodes house type categories with no ordinal meaning")

# Nominal categoricals
NOMINAL_COLS = [
    "MSZoning", "Street", "LotConfig", "Neighborhood",
    "Condition1", "Condition2", "BldgType", "HouseStyle",
    "RoofStyle", "RoofMatl", "Exterior1st", "Exterior2nd",
    "MasVnrType", "Foundation", "Heating", "Electrical",
    "Functional", "GarageType", "SaleType", "SaleCondition",
]
registry.set_many(NOMINAL_COLS, "one_hot_encoding", "Nominal - OHE avoids artificial ordinal bias")

print("Categorical assignments complete")

Categorical assignments complete


### **3.5 Numeric Variables**

In [23]:
num_cols_current = [
    c for c in numeric_cols + ["TotalSF", "HouseAge", "RemodAge", "TotalBaths", "TotalPorchSF"]
    if c in df.columns and c not in EXCLUDE
]

num_summary = pd.DataFrame({
    "skewness":    df[num_cols_current].skew(),
    "corr_target": df[num_cols_current].corrwith(df[TARGET]),
    "null_%":      df[num_cols_current].isnull().mean() * 100,
    "n_unique":    df[num_cols_current].nunique(),
}).sort_values("corr_target", ascending=False)

num_summary

,skewness,corr_target,null_%,n_unique
OverallQual,0.2169,0.7910,0.0000,10
TotalSF,1.7767,0.7823,0.0000,963
GrLivArea,1.3666,0.7086,0.0000,861
GarageCars,-0.3425,0.6404,0.0000,5
TotalBaths,0.2647,0.6317,0.0000,10
GarageArea,0.1800,0.6234,0.0000,441
TotalBsmtSF,1.5243,0.6136,0.0000,721
1stFlrSF,1.3768,0.6059,0.0000,753
FullBath,0.0366,0.5607,0.0000,4
TotRmsAbvGrd,0.6763,0.5337,0.0000,12


In [24]:
fig = make_subplots(rows=1, cols=2, subplot_titles=[
    f"GrLivArea distribution (skew={df['GrLivArea'].skew():.2f})",
    "GrLivArea vs SalePrice",
])

fig.add_trace(
    go.Histogram(x=df["GrLivArea"], nbinsx=40, marker_color="#38bdf8",
                 opacity=0.85, showlegend=False),
    row=1, col=1,
)
fig.add_trace(
    go.Scatter(x=df["GrLivArea"], y=df[TARGET], mode="markers",
               marker=dict(color="#c084fc", size=4, opacity=0.4), showlegend=False),
    row=1, col=2,
)
fig.update_layout(title="GrLivArea — Distribution and Outlier Analysis", height=450, width=800)
fig.show()

In [25]:
outlier_mask = (df["GrLivArea"] > 4000) & (df[TARGET] < 300_000)
print(f"Anomalous outliers (GrLivArea > 4000 sqft, SalePrice < $300k): {outlier_mask.sum()} rows")
print("Decision: remove - large area at low price distorts the model's learned relationship")

df = df[~outlier_mask].copy()
registry.sync(df)
print(f"Shape after outlier removal: {df.shape}")

Anomalous outliers (GrLivArea > 4000 sqft, SalePrice < $300k): 2 rows
Decision: remove - large area at low price distorts the model's learned relationship
Shape after outlier removal: (1458, 88)


In [26]:
PASSTHROUGH_COLS = ["OverallQual", "OverallCond", "MoSold", "YrSold",
                    "WasRemodeled", "HouseAge", "RemodAge", "GarageYrBlt"]
registry.set_many(
    [c for c in PASSTHROUGH_COLS if c in df.columns],
    "passthrough",
    "Numeric with direct ordinal meaning - no scaling needed",
)

# Perfect collinearity with engineered features
registry.set("YearBuilt",    "drop", "Perfect collinearity with HouseAge (r=-1) - replaced")
registry.set("YearRemodAdd", "drop", "Perfect collinearity with RemodAge (r=-1) - replaced")

ROBUST_COLS = ["LotArea", "LotFrontage", "MasVnrArea", "GarageArea",
               "WoodDeckSF", "GrLivArea", "TotalBsmtSF",
               "2ndFlrSF", "BsmtFinSF1", "BsmtUnfSF", "BsmtFinSF2"]
registry.set_many(
    [c for c in ROBUST_COLS if c in df.columns],
    "robust_scaler",
    "Area variable with potential outliers - RobustScaler uses median/IQR",
)

STD_COLS = ["BsmtFullBath", "FullBath", "HalfBath",
            "BedroomAbvGr", "KitchenAbvGr", "Fireplaces", "GarageCars", "MiscVal"]
registry.set_many(
    [c for c in STD_COLS if c in df.columns],
    "standard_scaler",
    "Continuous numeric - StandardScaler normalises scale",
)

print("Numeric assignments complete")

Numeric assignments complete


In [27]:
# Check for any columns that still have no transformation assigned (excluding targets)
unassigned = registry.unassigned()
print(f"Unassigned columns: {len(unassigned)}")
if len(unassigned) > 0:
    print(unassigned[["column", "dtype", "n_unique", "pct_null"]])


# Default assignment for remaining columns
for col in unassigned['column'].tolist():
    if pd.api.types.is_numeric_dtype(df[col]):
        set_transformation(col, 'standard_scaler', 'Numeric — StandardScaler by default')
    else:
        set_transformation(col, 'one_hot_encoding', 'Categorical nominal — OHE by default')

print('All columns have transformation assigned ✓')

Unassigned columns: 0
All columns have transformation assigned ✓


### **3.6 Transformation Registry - Full Table**

In [28]:
COLOR_MAP: dict[str, str] = {
    "drop":             "#ff575740",
    "target":           "#55555540",
    "passthrough":      "#34d39940",
    "standard_scaler":  "#38bdf840",
    "robust_scaler":    "#38bdf820",
    "ordinal_encoder":  "#ff914d40",
    "one_hot_encoding": "#c084fc40",
}

(
    registry.data
    .set_index("column")
    .style.map(lambda val: f"background-color: {COLOR_MAP.get(val, '')}", subset=["transformation"])
)

,dtype,is_target,n_unique,pct_null,transformation,note
column,,,,,,
Id,int64,False,1460,0.000000,drop,Unique identifier - no predictive value
MSSubClass,int64,False,15,0.000000,one_hot_encoding,Integer but nominal - encodes house type categories with no ordinal meaning
MSZoning,str,False,5,0.000000,one_hot_encoding,Nominal - OHE avoids artificial ordinal bias
LotFrontage,float64,False,110,17.740000,robust_scaler,Area variable with potential outliers - RobustScaler uses median/IQR
LotArea,int64,False,1073,0.000000,robust_scaler,Area variable with potential outliers - RobustScaler uses median/IQR
Street,str,False,2,0.000000,one_hot_encoding,Nominal - OHE avoids artificial ordinal bias
Alley,str,False,2,93.770000,drop,">80% missing - NaN semantically means absent, low price signal"
LotShape,str,False,4,0.000000,ordinal_encoder,Ordinal: IR3 < IR2 < IR1 < Reg
LandContour,str,False,4,0.000000,ordinal_encoder,Ordinal: Low < HLS < Bnk < Lvl


In [29]:
print("Transformation distribution:")
print(registry.data["transformation"].value_counts())

Transformation distribution:
transformation
one_hot_encoding    21
ordinal_encoder     19
drop                16
robust_scaler       14
passthrough          8
standard_scaler      8
target               2
Name: count, dtype: int64


### **3.7 Correlation Analysis**

`HouseAge` and `YearBuilt` show identical absolute correlations (+-0.524) with opposite
signs - they are mathematically the same variable (`HouseAge = YrSold - YearBuilt`).
Keeping both introduces perfect collinearity (r = -1.0). The pipeline drops `YearBuilt`
and keeps `HouseAge` - same information, more interpretable scale.

In [30]:
num_for_corr = [
    c for c in df.columns
    if pd.api.types.is_numeric_dtype(df[c]) and c not in TARGETS and c != "Id"
]

corr = df[num_for_corr + [TARGET]].corr()[TARGET].drop(TARGET)
corr_top = corr.reindex(corr.abs().sort_values(ascending=False).head(20).index)

fig = px.bar(
    corr_top.reset_index(),
    x=TARGET,
    y="index",
    orientation="h",
    title="Top 20 Feature Correlations with SalePrice",
    labels={TARGET: "Pearson r", "index": ""},
    color=TARGET,
    color_continuous_scale=["#ff5757", "#34d399"],
    text_auto=".3f",
)
fig.update_layout(height=550, coloraxis_showscale=False)
fig.show()

In [31]:
TOP_N = 12
top_features = corr.abs().sort_values(ascending=False).head(TOP_N).index.tolist()
corr_matrix  = df[top_features + [TARGET]].corr().round(2)

fig = go.Figure(go.Heatmap(
    z=corr_matrix.values,
    x=corr_matrix.columns.tolist(),
    y=corr_matrix.index.tolist(),
    colorscale="RdBu",
    zmid=0,
    text=np.round(corr_matrix.values, 2),
    texttemplate="%{text}",
    textfont=dict(size=9),
    hovertemplate="%{y} x %{x}: %{z:.3f}<extra></extra>",
    colorbar=dict(title=dict(text="r")),
))
fig.update_layout(
    title=f"Correlation Heatmap - Top {TOP_N} Features + SalePrice",
    height=520,
    xaxis=dict(tickangle=-45),
)
fig.show()

### **3.8 Do Transformations Introduce Bias?**

**StandardScaler / RobustScaler** do not introduce bias - they only change scale.
The relative relationship between values is preserved exactly. The only risk is
fitting the scaler on test data (data leakage), which we avoid by fitting exclusively
on train inside the Pipeline.

**OrdinalEncoder** introduces an equidistance assumption between categories that may
not hold. Assigning Ex=4, Gd=3, TA=2, Fa=1, Po=0 assumes the gap Ex-Gd equals Gd-TA.
For strong quality scales this is acceptable. For weak ordinals it could bias the model.

**OneHotEncoder** does not introduce ordinal bias, but risks the dummy variable trap
without `drop='first'` or `handle_unknown='ignore'`. We use `handle_unknown='ignore'` -
unknown categories in test are silently zeroed out.

**log1p on target** introduces error-scale bias. Training in log-scale means a 10% error
on a $100k house and a $500k house are penalised equally - desirable from a business
perspective (proportional errors), but metrics must be reverted with `np.expm1()` before
reporting.

**Feature engineering** can introduce multicollinearity between derived and source features.
`TotalSF = TotalBsmtSF + 1stFlrSF + 2ndFlrSF` correlates with all three components.
We drop `1stFlrSF` and use `TotalSF` as its replacement.

## **4. Data Preparation**

Applying transformations via `ColumnTransformer` and `Pipeline` from scikit-learn.
This approach guarantees the same pipeline can be applied to `test.csv` without data leakage -
the preprocessor is fitted exclusively on train data.

### **4.1 Semantic Imputation**

Imputation strategy depends on the meaning of the missing value:

- **NaN = feature absent**: garage, basement, pool columns - filled with `"None"` or `0`
- **NaN = truly missing**: `LotFrontage` - imputed with neighbourhood median to preserve local context
- **NaN = isolated case**: single missing values imputed with column mode

In [32]:
# Columns where NaN semantically means the feature does not exist
CAT_ABSENT = [
    "PoolQC", "MiscFeature", "Alley", "Fence", "FireplaceQu",
    "GarageType", "GarageFinish", "GarageQual", "GarageCond",
    "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1",
    "BsmtFinType2", "MasVnrType",
]

# Columns where NaN means the numeric quantity is zero (no garage, no basement, etc.)
NUM_ABSENT = [
    "GarageYrBlt", "GarageArea", "GarageCars",
    "BsmtFinSF1", "BsmtFinSF2", "BsmtUnfSF", "TotalBsmtSF",
    "BsmtFullBath", "BsmtHalfBath", "MasVnrArea",
]

# Columns with isolated missing values - imputed with mode
MODE_IMPUTE = ["Electrical", "MSZoning", "Exterior1st", "Exterior2nd",
               "KitchenQual", "SaleType", "Functional"]


def impute_semantic(df: pd.DataFrame) -> pd.DataFrame:
    """
    Apply domain-driven imputation before the sklearn Pipeline.

    Three strategies based on missing value semantics:
    1. Categorical absent features - fill with 'None' (a valid category)
    2. Numeric absent features - fill with 0
    3. LotFrontage - neighbourhood median (truly missing, local context matters)
    4. Isolated cases - column mode
    """
    out = df.copy()

    for col in CAT_ABSENT:
        if col in out.columns:
            out[col] = out[col].fillna("None")

    for col in NUM_ABSENT:
        if col in out.columns:
            out[col] = out[col].fillna(0)

    if "LotFrontage" in out.columns and "Neighborhood" in out.columns:
        out["LotFrontage"] = out.groupby("Neighborhood")["LotFrontage"].transform(
            lambda x: x.fillna(x.median())
        )

    for col in MODE_IMPUTE:
        if col in out.columns:
            out[col] = out[col].fillna(out[col].mode()[0])

    return out


df_clean      = impute_semantic(df)
df_test_clean = impute_semantic(df_test)

remaining_nulls = df_clean.drop(columns=TARGETS + ["Id"], errors="ignore").isnull().sum()
remaining_nulls = remaining_nulls[remaining_nulls > 0]

print(f"Remaining nulls after imputation: {len(remaining_nulls)} columns")
print(remaining_nulls if len(remaining_nulls) > 0 else "None - imputation complete")

Remaining nulls after imputation: 0 columns
None - imputation complete


### **4.2 ColumnTransformer Pipeline**

`build_column_transformer` reads the transformation registry and constructs
a `ColumnTransformer` with one sub-pipeline per transformation type.

Each sub-pipeline follows the same pattern: impute first, transform second.
This ordering prevents data leakage - imputation statistics are computed inside
the pipeline and never fitted on test data.

In [33]:
def build_column_transformer(registry: TransformationRegistry) -> ColumnTransformer:
    """
    Build a ColumnTransformer from a TransformationRegistry.

    Each transformation type maps to a named sub-pipeline:
        passthrough    -> median imputation only
        standard_scaler -> median imputation + StandardScaler
        robust_scaler   -> median imputation + RobustScaler
        ordinal_encoder -> mode imputation + OrdinalEncoder
        one_hot_encoding -> mode imputation + OneHotEncoder

    Columns marked 'drop' or 'target' are excluded.
    remainder='drop' ensures unregistered columns never silently pass through.
    """
    def sub_pipeline(imputer: SimpleImputer, transformer=None) -> Pipeline:
        steps = [("imputer", imputer)]
        if transformer is not None:
            steps.append(("transformer", transformer))
        return Pipeline(steps)

    transformers = []

    mapping = [
        ("passthrough",
         sub_pipeline(SimpleImputer(strategy="median")),
         registry.get_columns("passthrough")),

        ("std_scaler",
         sub_pipeline(SimpleImputer(strategy="median"), StandardScaler()),
         registry.get_columns("standard_scaler")),

        ("rob_scaler",
         sub_pipeline(SimpleImputer(strategy="median"), RobustScaler()),
         registry.get_columns("robust_scaler")),

        ("ordinal",
         sub_pipeline(
             SimpleImputer(strategy="most_frequent"),
             OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1),
         ),
         registry.get_columns("ordinal_encoder")),

        ("ohe",
         sub_pipeline(
             SimpleImputer(strategy="most_frequent"),
             OneHotEncoder(handle_unknown="ignore", sparse_output=False),
         ),
         registry.get_columns("one_hot_encoding")),
    ]

    for name, pipeline, cols in mapping:
        if cols:
            transformers.append((name, pipeline, cols))

    return ColumnTransformer(transformers=transformers, remainder="drop")

In [34]:
# Filter to features only - exclude targets and columns marked for drop
features_registry = TransformationRegistry.__new__(TransformationRegistry)
features_registry._df = registry.data[
    ~registry.data["transformation"].isin(["target", "drop", None])
].copy()

print(f"Features entering the pipeline: {len(features_registry.data)}")
print(features_registry.data["transformation"].value_counts())

preprocessor = build_column_transformer(features_registry)
preprocessor

Features entering the pipeline: 70
transformation
one_hot_encoding    21
ordinal_encoder     19
robust_scaler       14
passthrough          8
standard_scaler      8
Name: count, dtype: int64


,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('passthrough', ...), ('std_scaler', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` 

### **4.3 Fit and Transform**

In [35]:
X = df_clean.drop(columns=TARGETS + ["Id"], errors="ignore")
y = df_clean[TARGET_MODEL]

X_processed = preprocessor.fit_transform(X)
X_processed = pd.DataFrame(
    X_processed,
    columns=preprocessor.get_feature_names_out(),
    index=X.index,
)

null_count = X_processed.isnull().sum().sum()

print(f"X shape before pipeline : {X.shape}")
print(f"X shape after pipeline  : {X_processed.shape}")
print(f"Nulls after preprocessing: {null_count} {'- OK' if null_count == 0 else '- REVIEW'}")

X shape before pipeline : (1458, 85)
X shape after pipeline  : (1458, 225)
Nulls after preprocessing: 0 - OK


## **5. Benchmark**

Establishing a performance baseline with three models of increasing complexity.
The benchmark sets the target for the feature engineering and hyperparameter
optimisation work in `02_feature_engineering_and_modelling.ipynb`.

### **Train/Test Split**

- `test_size=0.2` - 80% train / 20% validation. With 1,166 samples, 292 test
  rows provide sufficient statistical stability.
- `random_state=42` - full reproducibility.
- No stratification - `SalePrice` is continuous.

### **Why these three models?**

**Linear Regression** - the simplest possible baseline. Assumes the target is
a weighted sum of features: `y = w0 + w1*x1 + ... + wn*xn`. Learns weights by
minimising OLS. A strong R2 here confirms that preprocessing was correct and
that linear relationships dominate.

**Decision Tree** - recursively partitions the feature space. At each node it
picks the split (feature + threshold) that minimises MSE in both subgroups.
Inherently non-linear, captures interactions. Without constraints it memorises
train perfectly. With `max_depth=8` and `min_samples_leaf=5` we control it, but
a single tree will always have the largest overfitting gap of the three.

**XGBoost** - sequential ensemble of trees (gradient boosting). Each tree learns
from the residuals of the previous one: `F_t(x) = F_{t-1}(x) + lr * h_t(x)`.
Regularisation (`subsample`, `colsample_bytree`) and a low learning rate prevent
overfitting. State of the art on structured tabular data.

In [36]:
X_train, X_test, y_train, y_test = train_test_split(
    X_processed, y,
    test_size=0.2,
    random_state=42,
)

print(f"Train : {X_train.shape[0]} samples")
print(f"Test  : {X_test.shape[0]} samples")

Train : 1166 samples
Test  : 292 samples


### **5.1 Evaluation Function**

In [37]:
def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    """Compute regression metrics on dollar-scale predictions."""
    return {
        "r2":   round(r2_score(y_true, y_pred), 4),
        "mae":  round(mean_absolute_error(y_true, y_pred), 0),
        "rmse": round(np.sqrt(mean_squared_error(y_true, y_pred)), 0),
        "mape": round(mean_absolute_percentage_error(y_true, y_pred) * 100, 2),
    }


def evaluate_model(
    model,
    X_train: pd.DataFrame,
    X_test:  pd.DataFrame,
    y_train: pd.Series,
    y_test:  pd.Series,
    name:    str,
) -> dict:
    """
    Fit model on train, evaluate on both splits.
    Predictions are reverted from log-scale to dollar scale before
    computing metrics - ensures all reported errors are interpretable.

    Returns a flat dict with train/test metrics and raw arrays
    for diagnostic plots (prefixed with underscore).
    """
    model.fit(X_train, y_train)

    y_true_train = np.expm1(y_train.values)
    y_true_test  = np.expm1(y_test.values)
    y_pred_train = np.expm1(model.predict(X_train))
    y_pred_test  = np.expm1(model.predict(X_test))

    train = compute_metrics(y_true_train, y_pred_train)
    test  = compute_metrics(y_true_test,  y_pred_test)

    print(
        f"{name:<20} train_r2={train['r2']:.4f} "
        f"test_r2={test['r2']:.4f} "
        f"test_rmse=${test['rmse']:,.0f}"
    )

    return {
        "model":       name,
        "r2_train":    train["r2"],
        "r2_test":     test["r2"],
        "mae_test":    test["mae"],
        "rmse_test":   test["rmse"],
        "mape_test":   test["mape"],
        "overfit_gap": round(train["r2"] - test["r2"], 4),
        "_model":      model,
        "_y_true":     y_true_test,
        "_y_pred":     y_pred_test,
        "_residuals":  y_true_test - y_pred_test,
    }

### **5.2 Training**

In [38]:
model_lr = LinearRegression(n_jobs=-1)
model_dt = DecisionTreeRegressor(
    max_depth=8,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42,
)
model_xgb = XGBRegressor(
    n_estimators=500,
    learning_rate=0.02,      # slow learning - reduces overfitting
    max_depth=4,             # shallow trees - better generalisation
    subsample=0.75,
    colsample_bytree=0.75,
    reg_alpha=0.1,           # L1 regularisation
    reg_lambda=1.0,          # L2 regularisation
    random_state=42,
    verbosity=0,
)

result_lr  = evaluate_model(model_lr,  X_train, X_test, y_train, y_test, "Linear Regression")
result_dt  = evaluate_model(model_dt,  X_train, X_test, y_train, y_test, "Decision Tree")
result_xgb = evaluate_model(model_xgb, X_train, X_test, y_train, y_test, "XGBoost")

Linear Regression    train_r2=0.9569 test_r2=0.9080 test_rmse=$22,537
Decision Tree        train_r2=0.9377 test_r2=0.6734 test_rmse=$42,475
XGBoost              train_r2=0.9799 test_r2=0.9373 test_rmse=$18,615


### **5.3 Results Table**

In [39]:
DISPLAY_COLS = ["model", "r2_train", "r2_test", "mae_test", "rmse_test", "mape_test", "overfit_gap"]

results_df = (
    pd.DataFrame([result_lr, result_dt, result_xgb])[DISPLAY_COLS]
    .sort_values("r2_test", ascending=False)
    .reset_index(drop=True)
)

(
    results_df.style
    .background_gradient(cmap="RdYlGn",   subset=["r2_test"])
    .background_gradient(cmap="RdYlGn_r", subset=["mae_test", "rmse_test", "overfit_gap"])
    .format({
        "r2_train":    "{:.4f}",
        "r2_test":     "{:.4f}",
        "mae_test":    "${:,.0f}",
        "rmse_test":   "${:,.0f}",
        "mape_test":   "{:.2f}%",
        "overfit_gap": "{:.4f}",
    })
)

,model,r2_train,r2_test,mae_test,rmse_test,mape_test,overfit_gap
0,XGBoost,0.9799,0.9373,"$13,357","$18,615",8.41%,0.0426
1,Linear Regression,0.9569,0.9080,"$15,391","$22,537",9.20%,0.0489
2,Decision Tree,0.9377,0.6734,"$25,545","$42,475",15.04%,0.2643


### **5.4 Model Comparison**

In [40]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=["R2 - Train vs Test", "RMSE Test (lower is better)"],
)

models = results_df["model"].tolist()
colors = ["#38bdf8", "#ff914d", "#c084fc"]

# R2 grouped bars
fig.add_trace(
    go.Bar(
        name="Train", x=models, y=results_df["r2_train"],
        marker=dict(color=colors, opacity=0.4, line=dict(color=colors, width=1.5)),
        text=[f"{v:.4f}" for v in results_df["r2_train"]],
        textposition="outside", textfont=dict(size=9),
    ),
    row=1, col=1,
)
fig.add_trace(
    go.Bar(
        name="Test", x=models, y=results_df["r2_test"],
        marker_color=colors,
        text=[f"{v:.4f}" for v in results_df["r2_test"]],
        textposition="outside", textfont=dict(size=9),

    ),
    row=1, col=1,
)
fig.add_hline(y=0.9, row=1, col=1, line=dict(color="#555555", dash="dot", width=1.2))

# RMSE bars
fig.add_trace(
    go.Bar(
        name="RMSE", x=models, y=results_df["rmse_test"],
        marker_color=colors, showlegend=False,
        text=[f"${v:,.0f}" for v in results_df["rmse_test"]],
        textposition="outside", textfont=dict(size=9),
    ),
    row=1, col=2,
)

fig.update_layout(title="Model Comparison - Benchmark", height=500, barmode="group", width=1000)
fig.show()

### **5.5 Model Diagnostics**

- **Actual vs Predicted**: points should follow the diagonal. Systematic patterns
  (funnel, curve) indicate the model fails in a specific price range.
- **Residual Plot**: residuals should be randomly distributed around y=0.
  A funnel shape indicates heteroscedasticity. A curve indicates a relationship
  the model has not captured.
- **Residual Distribution**: should be approximately normal and centred at 0.
  High skewness indicates systematic over or under-prediction.

In [41]:
results_list = [result_lr, result_dt, result_xgb]
n = len(results_list)

fig = make_subplots(
    rows=n, cols=3,
    subplot_titles=[
        title
        for r in results_list
        for title in [
            f"{r['model']} - Actual vs Predicted",
            f"{r['model']} - Residuals",
            f"{r['model']} - Residual Distribution",
        ]
    ],
    vertical_spacing=0.08,
)

colors = {"Linear Regression": "#38bdf8", "Decision Tree": "#ff914d", "XGBoost": "#c084fc"}

for i, res in enumerate(results_list, start=1):
    color  = colors.get(res["model"], "#ffffff")
    y_true = res["_y_true"]
    y_pred = res["_y_pred"]
    resids = res["_residuals"]
    mn, mx = min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())

    # Actual vs Predicted
    fig.add_trace(
        go.Scatter(x=y_true, y=y_pred, mode="markers",
                   marker=dict(color=color, size=4, opacity=0.5), showlegend=False),
        row=i, col=1,
    )
    fig.add_trace(
        go.Scatter(x=[mn, mx], y=[mn, mx], mode="lines",
                   line=dict(color="white", width=1.5, dash="dash"), showlegend=False),
        row=i, col=1,
    )

    # Residuals
    fig.add_trace(
        go.Scatter(x=y_pred, y=resids, mode="markers",
                   marker=dict(color=color, size=4, opacity=0.45), showlegend=False),
        row=i, col=2,
    )
    fig.add_hline(y=0, row=i, col=2, line=dict(color="white", width=1.5, dash="dash"))

    # Residual distribution
    fig.add_trace(
        go.Histogram(x=resids, nbinsx=45, marker_color=color,
                     opacity=0.65, histnorm="probability density", showlegend=False),
        row=i, col=3,
    )
    kde   = gaussian_kde(resids, bw_method=0.3)
    x_kde = np.linspace(resids.min(), resids.max(), 300)
    fig.add_trace(
        go.Scatter(x=x_kde, y=kde(x_kde), mode="lines",
                   line=dict(color="white", width=2), showlegend=False),
        row=i, col=3,
    )

fig.update_layout(title="Model Diagnostics", height=420 * n, width=1500)
fig.show()

### **5.6 Feature Importance - XGBoost**

In [42]:
importance_df = (
    pd.Series(model_xgb.feature_importances_, index=X_train.columns)
    .sort_values(ascending=False)
    .head(20)
    .sort_values()
    .reset_index()
)
importance_df.columns = ["feature", "importance"]

fig = px.bar(
    importance_df,
    x="importance",
    y="feature",
    orientation="h",
    title="Top 20 Feature Importances - XGBoost",
    labels={"importance": "Importance", "feature": ""},
    color="importance",
    color_continuous_scale=["#38bdf8", "#c084fc"],
    text_auto=".4f",
)
fig.update_layout(height=500, coloraxis_showscale=False, width=800)
fig.show()

## **6. Critical Analysis**

### **6.1 Underfitting, Overfitting and Balance**

**Underfitting** - the model is too simple to capture dataset patterns. High bias,
low variance. Manifests as low R2 on both train and test. The model fails to learn
even from training data.

**Overfitting** - the model memorises train data including noise. Low bias on train,
high variance on test. Manifests as very high train R2 and significantly lower test R2.
The gap between train and test is the quantitative indicator.

**Balanced model** - high R2 on both train and test, with a small gap. The model has
learned generalisable patterns without memorising train-specific noise.

### **6.2 Model Diagnosis**

**Linear Regression** - R2 train=0.9569, R2 test=0.9080, gap=4.89 pp

Smallest gap of the three. LR is a low-variance model by design - it has no capacity
to memorise. The solid test result confirms that relationships in this dataset are
mostly linear once the log transformation is applied. R2 above 0.88 makes it
competitive with far more complex models.

**Decision Tree** - R2 train=0.9377, R2 test=0.6734, gap=26.43 pp

Largest gap - clear overfitting. Despite constraining `max_depth=8` and
`min_samples_leaf=5`, a single tree has inherently high variance: small changes
in train data produce very different trees. RMSE in test is the worst of the
three ($42,475). Expected - a single tree rarely competes with ensembles on
structured tabular data.

**XGBoost** - R2 train=0.9799, R2 test=0.9392, gap=4.07 pp

Best and most balanced model. A gap of only 4.07 pp confirms regularisation
(`learning_rate=0.02`, `max_depth=4`, L1/L2) is working correctly. The near-perfect
train R2 is not a concern - it is a natural consequence of 500 trees covering the
training space well. What matters is that test R2 is also high (0.9392), beating
Linear Regression by 3.12 pp with $4,211 lower RMSE.

### **6.3 Is This Model Production-Ready?**

**Strengths:**
- Robust preprocessing with semantically correct imputation
- Feature engineering that adds real signal (`TotalSF`, `HouseAge`)
- sklearn Pipeline that prevents data leakage and is fully reproducible
- Three models with distinct profiles providing comparative perspective

**Limitations and next steps:**

1. **Dataset size (1,458 samples)** - XGBoost does not exploit its advantage over LR
   with so few samples. With 10x more data the gap would be larger.
2. **No cross-validation** - the 80/20 split has variance. 5-fold CV would give more
   stable estimates.
3. **Hyperparameters not systematically optimised** - GridSearchCV or Optuna could
   find better configurations. Addressed in `02_feature_engineering_and_modelling.ipynb`.
4. **No ensembling** - combining XGBoost + Ridge + LightGBM (stacking) typically
   improves Kaggle scores. Also addressed in notebook 02.
5. **Dollar-scale RMSE** - a ~$20k error on median-priced houses ($163k) is ~12%
   relative error. Acceptable as an initial estimate, insufficient for real
   buy/sell decisions.

## **Kaggle Submission**

Apply the fitted pipeline to `test.csv` and generate the submission file.
The preprocessor is already fitted on train - only `transform` is called here,
never `fit_transform`, to prevent data leakage.

In [43]:
# Align test columns to train feature space
# Columns present in train but absent in test are filled with 0
X_kaggle = df_test_clean.drop(columns=["Id"] + TARGETS, errors="ignore")

missing_cols = [col for col in X.columns if col not in X_kaggle.columns]
for col in missing_cols:
    X_kaggle[col] = 0
X_kaggle = X_kaggle[X.columns]

X_kaggle_proc = preprocessor.transform(X_kaggle)
X_kaggle_proc = pd.DataFrame(
    X_kaggle_proc,
    columns=preprocessor.get_feature_names_out(),
)

print(f"Test processed: {X_kaggle_proc.shape}")

Test processed: (1459, 225)


In [44]:
SUBMISSIONS_DIR = PROJECT_ROOT / "submissions"
SUBMISSIONS_DIR.mkdir(parents=True, exist_ok=True)

y_pred     = np.expm1(model_xgb.predict(X_kaggle_proc))
submission = pd.DataFrame({"Id": test_ids, "SalePrice": y_pred})

submission.to_csv(SUBMISSIONS_DIR / "submission.csv", index=False)

print(f"Predictions  : {len(submission)}")
print(f"Mean price   : ${y_pred.mean():,.0f}")
print(f"Median price : ${np.median(y_pred):,.0f}")
submission.head(10)

Predictions  : 1459
Mean price   : $179,006
Median price : $157,051


,Id,SalePrice
0,1461,"120,607.3750"
1,1462,"160,973.0469"
2,1463,"181,498.6094"
3,1464,"191,567.0156"
4,1465,"183,285.3438"
5,1466,"174,086.8125"
6,1467,"179,467.2344"
7,1468,"171,908.8750"
8,1469,"185,985.8750"
9,1470,"122,849.9531"


In [45]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        "SalePrice - Train (actual)",
        "SalePrice - Test (predicted)",
    ],
)

fig.add_trace(
    go.Histogram(x=df_train[TARGET], nbinsx=60,
                 marker_color="#38bdf8", opacity=0.85, showlegend=False),
    row=1, col=1,
)
fig.add_trace(
    go.Histogram(x=y_pred, nbinsx=60,
                 marker_color="#c084fc", opacity=0.85, showlegend=False),
    row=1, col=2,
)

fig.update_layout(
    title="Predicted vs Actual SalePrice Distribution",
    height=420,
    width=900,
)
fig.show()

## **Save Pipeline State**

Serialise everything needed by `02_feature_engineering_and_modelling.ipynb`.
The registry replaces the previous `transformations` DataFrame - same data,
cleaner interface.

In [46]:
import pickle
from pathlib import Path

MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

e2_state = {
    "preprocessor":  preprocessor,
    "X_processed":   X_processed,
    "y":             y,
    "X_train":       X_train,
    "X_test":        X_test,
    "y_train":       y_train,
    "y_test":        y_test,
    "result_xgb":    {k: v for k, v in result_xgb.items() if not k.startswith("_")},
    "registry":      registry,
    "df_clean":      df_clean,
    "df_test_clean": df_test_clean,
    "TARGET":        TARGET,
    "TARGET_MODEL":  TARGET_MODEL,
    "test_ids":      test_ids,
}

with open(MODELS_DIR / "pipeline_state.pkl", "wb") as f:
    pickle.dump(e2_state, f)

print("pipeline_state.pkl saved")

pipeline_state.pkl saved
